In [11]:
import pandas as pd
import numpy as np
import ast
import os
import json
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# 1. Get the embedding of each PSOC task

In [12]:
# Open up the data of the mca data with the SOC codes and the json of tasks
mca_df = pd.read_csv('../data/auxiliary/mca_soc_codes.csv', dtype={'PSOC Code':str, 'ISCO Code':str})
mca_df['SOC Codes'] = mca_df['SOC Codes'].apply(ast.literal_eval)
mca_df['Educational Qualification'] = (
    mca_df['Educational Qualification'].str.strip().replace({'No Match Found': ''})
)

with open('../data/auxiliary/psoc_tasks_map.json', 'r') as file:
    psoc_tasks_map = json.load(file)

with open('../data/auxiliary/embedded_soc_tasks_map.json', 'r') as file:
    embedded_soc_tasks_map = json.load(file)

In [13]:
# Map the PSOC tasks and the embedded SOC tasks
mca_df['PSOC Tasks'] = mca_df['PSOC Code'].map(psoc_tasks_map)

def get_embedded_soc_tasks(codes, embedded_soc_tasks_map):
    """Get embedded SOC tasks for available SOC codes."""

    embedded_tasks = []

    for code in codes:
        # Exact match
        if code in embedded_soc_tasks_map:
            embedded_tasks.append(embedded_soc_tasks_map[code])
            continue

        # Fallback: same first 4 digits
        prefix = code[:4]

        for soc_code, tasks in embedded_soc_tasks_map.items():
            if soc_code[:4] == prefix:
                embedded_tasks.append(tasks)

    return embedded_tasks

# embed the SOC Codes
mca_df['Embedded SOC Codes'] = mca_df['SOC Codes'].apply(
    get_embedded_soc_tasks,
    embedded_soc_tasks_map=embedded_soc_tasks_map
)

# create PSOC tasks that have context by adding the job title
mca_df['PSOC Tasks with Context'] = mca_df.apply(
    lambda job: [
        f"{job['Job Title']}, "
        f"{task}"
        for task in job['PSOC Tasks']
    ] if isinstance(job['PSOC Tasks'], list) else job['PSOC Tasks'],
    axis=1
)

In [14]:
output_path = "../data/auxiliary/mca_soc_embedded.csv"

if os.path.exists(output_path):
    mca_df = pd.read_csv(output_path)
    # Convert the saved string representations back into lists
    mca_df['Embedded PSOC Tasks with Context'] = (
        mca_df['Embedded PSOC Tasks with Context']
        .apply(ast.literal_eval)
    )

    # deal with the annoying fact that CSVs dont save lists
    mca_df['SOC Codes'] = (
        mca_df['SOC Codes']
        .apply(ast.literal_eval)
    )

    mca_df['Embedded SOC Codes'] = mca_df['SOC Codes'].apply(
        get_embedded_soc_tasks,
        embedded_soc_tasks_map=embedded_soc_tasks_map
    )

else:
    embedding_model = SentenceTransformer("all-mpnet-base-v2")

    # create the embeddings per task for each row
    mca_df['Embedded PSOC Tasks with Context'] = mca_df[
        'PSOC Tasks with Context'
    ].apply(
        lambda tasks_with_context: [
            embedding_model.encode(
                task_with_context,
                normalize_embeddings=True
            ).tolist()
            for task_with_context in tasks_with_context
        ] if isinstance(tasks_with_context, list) else tasks_with_context
    )
    mca_df.to_csv(output_path, index=False)

In [15]:
# embed also the names of the job and of the SOC codes
soc_df = pd.read_csv(
    '../data/labor_codes/2019_to_SOC_Crosswalk.csv',
    usecols=[0, 1],
    names=['SOC', 'Title'], 
    skiprows=1
)

soc_code_title_map = dict(zip(soc_df.SOC, soc_df.Title))

In [16]:
# make sure SOC Codes is actually filled with lists and NOT a string
mca_df['SOC Titles'] = mca_df['SOC Codes'].apply(
    lambda codes : [soc_code_title_map[code] for code in codes]
)

# Embed the Job Title and the SOC Codes
embedding_model = SentenceTransformer("all-mpnet-base-v2")
mca_df['Embedded Job Title'] = mca_df['Job Title'].apply(
    lambda job_title : [embedding_model.encode(job_title, normalize_embeddings=True).tolist()]
)

mca_df['Embedded SOC Titles'] = mca_df['SOC Titles'].apply(
    lambda SOC_Titles: [
        embedding_model.encode(
            SOC_Title,
            normalize_embeddings=True
        ).tolist()
        for SOC_Title in SOC_Titles
    ])

# 2. Choose the most representative SOC code for each Occupation based on Cosine Similarity

For each PSOC occupation, we compare its tasks and job title against those of each candidate SOC code using cosine similarity between their embeddings.

For the tasks, we find the most similar SOC task for each PSOC task and take the mean of these best-match scores. Separately, we calculate the cosine similarity between the PSOC job title and the SOC title.

We then take the mean of the task similarity and title similarity to obtain the overall PSOC-SOC similarity score.

**PSOC tasks + job title -> task similarity + title similarity -> average -> representative SOC code**

A higher overall score indicates that the SOC is more semantically representative of the PSOC occupation.

In [17]:
# def estimate_representative_soc_code(job, verbose=False):
#     """
#     Estimate the most representative SOC code for a PSOC occupation.

#     Each candidate SOC code is evaluated using:
#     1. Task similarity: mean of the best cosine similarity for each
#        PSOC task against the candidate SOC's tasks.
#     2. Title similarity: cosine similarity between the PSOC job title
#        and the candidate SOC title.

#     The final score is the mean of the task and title similarities.
#     """


#     # set default variables that will be used for getting most representative
#     highest_mean_similarity = -np.inf
#     representative_soc_code = None

#     # convert to np.array the embedded psoc tasks and job title
#     embedded_psoc_tasks = np.array(
#         job['Embedded PSOC Tasks with Context']
#     )
#     embedded_job_title = np.array(
#         job['Embedded Job Title']
#     ).reshape(1, -1)

#     # now go thrugh each possible SOC code in SOC codes
#     for embedded_soc_tasks, embedded_soc_title, soc_code in zip(
#         job['Embedded SOC Codes'],
#         job['Embedded SOC Titles'],
#         job['SOC Codes']
#     ):
#         # focus on tasks
#         soc_tasks_array = np.array(embedded_soc_tasks)

#         similarity_matrix = cosine_similarity(
#             embedded_psoc_tasks,
#             soc_tasks_array
#         )

#         best_similarity_scores = similarity_matrix.max(axis=1)

#         task_similarity = best_similarity_scores.mean()

#         # focus on title
#         soc_title_embedding = np.array(
#             embedded_soc_title
#         ).reshape(1, -1)

#         title_similarity = cosine_similarity(
#             embedded_job_title,
#             soc_title_embedding
#         )[0, 0]

#         # get mean similarity
#         mean_similarity = (task_similarity + title_similarity) / 2

#         if verbose:
#             print(
#                 f"{soc_code}: "
#                 f"Task = {task_similarity:.4f}, "
#                 f"Title = {title_similarity:.4f}, "
#                 f"Average = {mean_similarity:.4f}"
#             )

#         # set if the most representative
#         if mean_similarity >= highest_mean_similarity:
#             highest_mean_similarity = mean_similarity
#             representative_soc_code = soc_code

#     return representative_soc_code

In [18]:
def estimate_representative_soc_codes(job, verbose=False):
    """
    Rank candidate SOC codes from most to least representative.

    Each candidate SOC code is evaluated using:
    1. Task similarity: mean of the best cosine similarity for each
       PSOC task against the candidate SOC's tasks.
    2. Title similarity: cosine similarity between the PSOC job title
       and the candidate SOC title.

    The final score is the mean of the task and title similarities.

    Returns:
        List of tuples: [(SOC code, score), ...], sorted descending by score.
    """

    # Convert embedded PSOC tasks and job title
    embedded_psoc_tasks = np.array(
        job['Embedded PSOC Tasks with Context']
    )

    embedded_job_title = np.array(
        job['Embedded Job Title']
    ).reshape(1, -1)

    soc_scores = []

    # Evaluate every candidate SOC code
    for embedded_soc_tasks, embedded_soc_title, soc_code in zip(
        job['Embedded SOC Codes'],
        job['Embedded SOC Titles'],
        job['SOC Codes']
    ):
        # Task similarity
        soc_tasks_array = np.array(embedded_soc_tasks)

        similarity_matrix = cosine_similarity(
            embedded_psoc_tasks,
            soc_tasks_array
        )

        best_similarity_scores = similarity_matrix.max(axis=1)

        task_similarity = best_similarity_scores.mean()

        # Title similarity
        soc_title_embedding = np.array(
            embedded_soc_title
        ).reshape(1, -1)

        title_similarity = cosine_similarity(
            embedded_job_title,
            soc_title_embedding
        )[0, 0]

        # Final score
        mean_similarity = (
            task_similarity + title_similarity
        ) / 2

        soc_scores.append({
            'SOC Code': soc_code,
            'Task Similarity': task_similarity,
            'Title Similarity': title_similarity,
            'Score': mean_similarity
        })

    # Sort from most likely to least likely
    soc_scores = sorted(
        soc_scores,
        key=lambda x: x['Score'],
        reverse=True
    )

    if verbose:
        for rank, result in enumerate(soc_scores, start=1):
            print(
                f"{rank}. {result['SOC Code']}: "
                f"Task = {result['Task Similarity']:.4f}, "
                f"Title = {result['Title Similarity']:.4f}, "
                f"Average = {result['Score']:.4f}"
            )

    return soc_scores

In [ ]:
# get the most representative SOC code
#mca_df['SOC Code'] = mca_df.apply(estimate_representative_soc_codes, axis=1)
mca_df['SOC Title'] = mca_df['SOC Code'].map(soc_code_title_map)

TypeError: unhashable type: 'list'

In [ ]:
relevant_cols = [
    'Job Title',
    'Educational Qualification',
    'Job Sector',
    'Educational Pathway',
    'HEI with PRC Exam',
    'Some HEI', 
    'Job Subsector',
    'PSOC Code',
    'ISCO Code',
    'SOC Code',
    'SOC Title'
]
mca_df[relevant_cols].to_csv('../data/auxiliary/final_mca_soc_code.csv', index=False)

The two examples below demonstrate how accurately the system identifies the most representative SOC code. 

For example, the teachers in the first table had 37 possible SOC codes to choose from, while the occupational therapist in the second table had 19. However, the previous approach would naively assign the same general SOC code repeatedly across the different teacher occupations, which is as 25-1199.00 (Postsecondary Teachers, All Other). This results in a loss of variety and specificity in the mappings. Similarly, the occupational therapist could be assigned the general code 31-9099.00 (Healthcare Support Workers, All Other). 

With the new system, the combination of task and title similarity allows the system to distinguish between occupations and select more specific and representative SOC codes from the available candidates.

In [ ]:
is_37_SOC_codes = (mca_df['SOC Codes'].apply(len) == 37)
mca_df.loc[is_37_SOC_codes, ['Job Title', 'SOC Code', 'SOC Title']]

,Job Title,SOC Code,SOC Title
103,Ethics Research Assistant,25-9044.00,"Teaching Assistants, Postsecondary"
110,Development Researcher,25-1192.00,"Family and Consumer Sciences Teachers, Postsec..."
126,Humanities Researcher,25-1125.00,"History Teachers, Postsecondary"
129,Legislative Researcher,25-1065.00,"Political Science Teachers, Postsecondary"
197,Religious Research Assistant,25-1126.00,"Philosophy and Religion Teachers, Postsecondary"
506,Anatomy and Physiology Instructor,25-1071.00,"Health Specialties Teachers, Postsecondary"
514,Biology Instructor,25-1042.00,"Biological Science Teachers, Postsecondary"
645,Mathematics Instructor,25-1022.00,"Mathematical Science Teachers, Postsecondary"
673,Philosophy and Ethics Instructor,25-1126.00,"Philosophy and Religion Teachers, Postsecondary"


In [ ]:
is_19_SOC_codes = (mca_df['SOC Codes'].apply(len) == 19)
mca_df.loc[is_19_SOC_codes, ['Job Title', 'SOC Code', 'SOC Title']]

,Job Title,SOC Code,SOC Title
662,Occupational Therapy Assistant,31-2011.00,Occupational Therapy Assistants


In [ ]:
is_11_SOC_codes = (mca_df['SOC Codes'].apply(len) == 19)
mca_df.loc[is_11_SOC_codes, ['Job Title', 'SOC Code', 'SOC Title']]

,Job Title,SOC Title
662,Occupational Therapy Assistant,Occupational Therapy Assistants


In [ ]:
mca_df.loc[[3, 4], ['Job Title', 'SOC Code', 'SOC Title']]

,Job Title,SOC Code,SOC Title
3,Financial Auditor,13-2011.00,Accountants and Auditors
4,Tax Accountant,13-2082.00,Tax Preparers


# 4. Mapping SOC to AIOE

In [ ]:
# read the mca data with SOC codes and the SOC-AIOE mapping
#mca_df = pd.read_csv('../data/auxiliary/final_mca_soc_code.csv')
soc_aioe_df = pd.read_csv('../data/ai_measurements/soc_aioe.csv')
soc_aioe_map = dict(zip(soc_aioe_df.SOC, soc_aioe_df.AIOE))

mca_df['AIOE'] = mca_df['SOC Code'].map(soc_aioe_map)